# Projeto Germinacao - Treino YOLO11 no Colab

Adaptado do `train.py` local (que rodava em MPS no Mac) para CUDA no Colab Pro.

**Antes de rodar:** Runtime > Change runtime type > GPU (T4 ou L4, dependendo da disponibilidade).

**Resiliencia:** checkpoints sao salvos direto no Google Drive. Se a sessao cair, basta abrir o notebook de novo, montar o Drive e usar a celula de **resume** (final do notebook).

## 1. Verificar GPU

In [ ]:
!nvidia-smi

## 2. Montar Google Drive (checkpoints persistem aqui)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUNS_DIR = '/content/drive/MyDrive/projetogerminacao/runs/train'
os.makedirs(RUNS_DIR, exist_ok=True)
print(f'Runs vao para: {RUNS_DIR}')

## 3. Clonar repo (codigo + dataset)

In [ ]:
!rm -rf /content/repo
!git clone --depth 1 https://github.com/nikolasdehor/projetogerminacao.git /content/repo
%cd /content/repo
!ls dataset/train/images | wc -l
!ls dataset/valid/images | wc -l
!ls dataset/test/images | wc -l

## 4. Instalar dependencias

Versoes pinadas para evitar surpresa em rerun. Ultralytics atualiza rapido.

In [ ]:
!pip install -q 'ultralytics>=8.3,<8.4' 'torch>=2.3' 'torchvision>=0.18' 'opencv-python-headless>=4.10' 'Pillow>=10.4' 'numpy>=1.26' 'pyyaml'

## 5. Detectar VRAM e definir batch dinamicamente

- T4 16GB -> batch 8 (igual ao Mac)
- L4 24GB -> batch 16 (acelera ~40 porcento)
- A100 40GB -> batch 24

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.1f}GB VRAM)')

if vram_gb >= 35:
    BATCH = 24
elif vram_gb >= 20:
    BATCH = 16
else:
    BATCH = 8

print(f'Batch escolhido: {BATCH}')

## 6. Preparar `data_train.yaml` com paths do Colab

In [ ]:
from pathlib import Path
import yaml

BASE = Path('/content/repo')
DATASET = BASE / 'dataset'
DATA_YAML = DATASET / 'data.yaml'
FIXED_YAML = BASE / 'data_train.yaml'

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

cfg['train'] = str((DATASET / 'train/images').resolve())
cfg['val']   = str((DATASET / 'valid/images').resolve())
cfg['test']  = str((DATASET / 'test/images').resolve())

with open(FIXED_YAML, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print('Config final:')
print(open(FIXED_YAML).read())

## 7. Smoke test (2 epochs, valida pipeline antes do treino real)

Roda em ~5-10 min. Confirma que dataset carrega, loss desce, sem OOM. Se passar, pula pra celula 8.

In [ ]:
from datetime import datetime
from ultralytics import YOLO

model = YOLO('yolo11s.pt')
smoke = model.train(
    data=str(FIXED_YAML),
    epochs=2,
    imgsz=1280,
    batch=BATCH,
    device=0,
    workers=4,
    project='/content/smoke',
    name='smoke_test',
    exist_ok=True,
    cache=False,
    cos_lr=True,
)
print('\nSmoke test OK. Pode rodar o treino real.')

## 8. Treino real (100 epochs, salva no Drive)

Hiperparametros identicos ao `train.py` local + `cos_lr=True` (cosine LR scheduler, melhor pra runs longos).

**Tempo estimado:**
- T4 (batch 8): 5-7 horas
- L4 (batch 16): 3-4 horas
- A100 (batch 24): 1.5-2 horas

Patience=20 vai pausar cedo se mAP estagnar.

In [ ]:
from datetime import datetime
from ultralytics import YOLO

RUN_NAME = f"train_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f'Run name: {RUN_NAME}')
print(f'Salvando em: {RUNS_DIR}/{RUN_NAME}')

model = YOLO('yolo11s.pt')
results = model.train(
    data=str(FIXED_YAML),
    epochs=100,
    imgsz=1280,
    batch=BATCH,
    patience=20,
    device=0,
    workers=4,
    project=RUNS_DIR,
    name=RUN_NAME,
    exist_ok=False,
    cache=False,
    resume=False,
    cos_lr=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.3,
    shear=0.0, perspective=0.0,
    flipud=0.3, fliplr=0.5,
    mosaic=0.5, mixup=0.0, copy_paste=0.0,
    plots=True,
    save=True,
    save_period=5,
)

best_src = Path(results.save_dir) / 'weights' / 'best.pt'
if best_src.exists():
    size_mb = best_src.stat().st_size / 1e6
    print(f'\nMelhor modelo: {best_src} ({size_mb:.1f}MB)')
else:
    print(f'\nbest.pt nao encontrado em {best_src}')

## 9. Validar modelo no conjunto de teste

In [ ]:
from ultralytics import YOLO

best = Path(results.save_dir) / 'weights' / 'best.pt'
model = YOLO(str(best))
metrics = model.val(data=str(FIXED_YAML), split='test', imgsz=1280, device=0)
print(f'mAP@0.5: {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')

## 10. Resume (use se a sessao cair durante o treino)

Edite `LAST_RUN_NAME` para o nome do run que parou (esta em `/content/drive/MyDrive/projetogerminacao/runs/train/`).

In [ ]:
# from ultralytics import YOLO
# LAST_RUN_NAME = 'train_YYYYMMDD_HHMMSS'
# last_ckpt = f'{RUNS_DIR}/{LAST_RUN_NAME}/weights/last.pt'
# model = YOLO(last_ckpt)
# model.train(resume=True)

## 11. (Opcional) Baixar best.pt pro Mac

Quando terminar, baixe o modelo treinado pra usar local.

In [ ]:
# from google.colab import files
# files.download(str(best))